# Meyer–Wallach Entanglement on IQM Spark

**Authors:** Koło Naukowe Axion

**Goal:** measure **Meyer–Wallach (MW) entanglement** on the IQM Spark (Odra) quantum computer without statevectors, using **local X/Y/Z Pauli tomography** per qubit.

**Protocol**
1. For each ansatz × depth × random parameter sample, bind random $\theta \in [0, 2\pi)$.
2. Run **3 circuits** per sample (Z, X, Y basis rotations + `measure_all`).
3. From counts, estimate $\langle X\rangle$, $\langle Y\rangle$, $\langle Z\rangle$ per qubit; compute per-qubit purity and MW score.
4. Aggregate mean/std/sem/min/max over samples.

**Scale:** 2 ansatze × 3 depths × 20 samples × 3 bases = **360 circuits** (4096 shots each) by default.


## 1. Imports & Configuration

In [ ]:
import csv
import getpass
import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector

try:
    from iqm.qiskit_iqm import transpile_to_IQM as _iqm_transpile
    from iqm.qiskit_iqm.iqm_backend import IQMBackendBase as _IQMBackendBase
except ImportError:
    _iqm_transpile = None
    _IQMBackendBase = None

NUM_QUBITS = 5
DEPTHS = [2, 4, 6]
ANSATZES = ("ansatz_odra", "ansatz_simulator")
SEED = 42
N_SAMPLES = 20
SHOTS = 4096
OPTIMIZATION_LEVEL = 1
MAX_CIRCUITS_PER_JOB = 275
IQM_URL = os.environ.get("IQM_URL", "https://odra5.e-science.pl/").strip()
NOTEBOOK_DIR = Path(".").resolve()

n_jobs = len(ANSATZES) * len(DEPTHS) * N_SAMPLES * 3
print(
    f"Planned run: {len(ANSATZES)} ansatze x {len(DEPTHS)} depths x "
    f"{N_SAMPLES} samples x 3 bases = {n_jobs} circuits ({SHOTS} shots each)"
)


## 2. Ansatz Definitions

- **Ansatz 1 — Ring (simulator-optimized).** RY · CRX · RX · CRY ring with a q0-incident reverse trim on the last macro-layer.
- **Ansatz 2 — Odra (IQM-Spark adapted).** RY · RZ/CZ · RX · RY/CZ ring with the same trim.


In [ ]:
def ansatz_trimmed_reverse_q0_param_count(n_qubits: int, depth: int) -> int:
    """Weights when only the last macro-layer uses the q0-incident reverse trim."""
    n_macro = depth // 2
    if n_macro == 0:
        return 0
    full = 4 * n_qubits
    last = 3 * n_qubits + 2
    return (n_macro - 1) * full + last


def ansatz_odra(n_qubits: int, depth: int) -> QuantumCircuit:
    n_macro = depth // 2
    theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
    qc = QuantumCircuit(n_qubits)
    p = 0

    for j in range(n_macro):
        last_layer = j == n_macro - 1

        for i in range(n_qubits):
            qc.ry(theta[p + i], i)
        p += n_qubits

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.rz(theta[p + i], target)
            qc.cz(control, target)
        p += n_qubits

        for i in range(n_qubits):
            qc.rx(theta[p + i], i)
        p += n_qubits

        if last_layer:
            for k in range(2):
                i = k
                control = i
                target = (i - 1) % n_qubits
                qc.ry(theta[p + k], target)
                qc.cz(control, target)
            p += 2
        else:
            for i in range(n_qubits):
                control = i
                target = (i - 1) % n_qubits
                qc.ry(theta[p + i], target)
                qc.cz(control, target)
            p += n_qubits

    assert p == len(theta)
    return qc


def ansatz_simulator(n_qubits: int, depth: int) -> QuantumCircuit:
    n_macro = depth // 2
    theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
    qc = QuantumCircuit(n_qubits)
    param_idx = 0

    for j in range(n_macro):
        last_layer = j == n_macro - 1

        for i in range(n_qubits):
            qc.ry(theta[param_idx], i)
            param_idx += 1

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.crx(theta[param_idx], control, target)
            param_idx += 1

        for i in range(n_qubits):
            qc.rx(theta[param_idx], i)
            param_idx += 1

        if last_layer:
            for k in range(2):
                i = k
                control = i
                target = (i - 1) % n_qubits
                qc.cry(theta[param_idx], control, target)
                param_idx += 1
        else:
            for i in range(n_qubits):
                control = i
                target = (i - 1) % n_qubits
                qc.cry(theta[param_idx], control, target)
                param_idx += 1

    assert param_idx == len(theta)
    return qc


def single_qubit_reduced_density(
    state: np.ndarray, qubit: int, n_qubits: int
) -> np.ndarray:
    arr = state.reshape((2,) * n_qubits)
    arr = np.moveaxis(arr, qubit, 0)
    psi_mat = arr.reshape(2, -1)
    return psi_mat @ psi_mat.conj().T


def meyer_wallach_score(state: np.ndarray, n_qubits: int) -> float:
    if n_qubits < 1:
        return 0.0

    acc = 0.0
    for i in range(n_qubits):
        rho_i = single_qubit_reduced_density(state, i, n_qubits)
        purity = float(np.real(np.trace(rho_i @ rho_i)))
        acc += 1.0 - purity

    q = (2.0 / n_qubits) * acc
    return float(max(0.0, min(1.0, q)))


def verify_mw_implementation() -> None:
    n = 3
    product = np.zeros(2**n, dtype=complex)
    product[0] = 1.0
    assert abs(meyer_wallach_score(product, n)) < 1e-9

    ghz = np.zeros(2**n, dtype=complex)
    ghz[0] = 1.0 / math.sqrt(2.0)
    ghz[-1] = 1.0 / math.sqrt(2.0)
    assert abs(meyer_wallach_score(ghz, n) - 1.0) < 1e-9


for depth in DEPTHS:
    n_params = ansatz_trimmed_reverse_q0_param_count(NUM_QUBITS, depth)
    print(f"depth={depth}: n_params={n_params}")


## 3. Meyer–Wallach & Measurement Helpers

In [ ]:
BasisName = Literal["Z", "X", "Y"]
BASIS_ORDER: tuple[BasisName, ...] = ("Z", "X", "Y")

DEFAULT_IQM_URL = "https://odra5.e-science.pl/"
DEFAULT_SHOTS = 4096
DEFAULT_N_SAMPLES = 20
DEFAULT_SEED = 42
DEFAULT_OPTIMIZATION_LEVEL = 1
DEFAULT_MAX_CIRCUITS_PER_JOB = 275


def connect_to_iqm_backend(iqm_url: str, token: str | None = None):
    env_token = os.environ.get("IQM_TOKEN", "").strip()
    if token and env_token:
        raise ValueError("Set either --iqm-token or IQM_TOKEN, not both")
    if token is None and not env_token:
        token = getpass.getpass("Enter IQM Token: ").strip()
    from iqm.qiskit_iqm import IQMProvider

    if token:
        provider = IQMProvider(iqm_url, token=token)
    else:
        provider = IQMProvider(iqm_url)
    return provider.get_backend()


def transpile_for_backend(circuit: QuantumCircuit, backend, optimization_level: int, seed_transpiler: int | None):
    kwargs: dict[str, object] = {"optimization_level": optimization_level}
    if seed_transpiler is not None:
        kwargs["seed_transpiler"] = seed_transpiler
    if (
        _iqm_transpile is not None
        and _IQMBackendBase is not None
        and isinstance(backend, _IQMBackendBase)
    ):
        return _iqm_transpile(circuit, backend, **kwargs)
    return transpile(circuit, backend, **kwargs)


def bitstring_qubit_value(bitstring: str, qubit: int, n_qubits: int) -> str:
    """Return '0' or '1' for logical qubit index (0 = Qiskit LSB / rightmost bit)."""
    if len(bitstring) != n_qubits:
        raise ValueError(f"Expected bitstring length {n_qubits}, got {len(bitstring)!r}")
    return bitstring[-(qubit + 1)]


def qubit_expectation_from_counts(counts: dict[str, int], qubit: int, n_qubits: int) -> float:
    """Estimate <Pauli> for one qubit from computational-basis counts after basis rotation."""
    shots = sum(counts.values())
    if shots == 0:
        return 0.0
    expval = 0.0
    for bitstring, count in counts.items():
        bit = bitstring_qubit_value(bitstring, qubit, n_qubits)
        eigenvalue = 1.0 if bit == "0" else -1.0
        expval += eigenvalue * count / shots
    return float(expval)


def mw_score_from_bloch(
    x_expectations: list[float],
    y_expectations: list[float],
    z_expectations: list[float],
) -> float:
    n_qubits = len(x_expectations)
    if n_qubits < 1:
        return 0.0
    acc = 0.0
    for x_i, y_i, z_i in zip(x_expectations, y_expectations, z_expectations):
        purity = 0.5 * (1.0 + x_i * x_i + y_i * y_i + z_i * z_i)
        acc += 1.0 - purity
    q = (2.0 / n_qubits) * acc
    return float(max(0.0, min(1.0, q)))


def add_basis_measurement(circuit: QuantumCircuit, basis: BasisName) -> QuantumCircuit:
    qc = circuit.copy()
    if basis == "X":
        for i in range(qc.num_qubits):
            qc.h(i)
    elif basis == "Y":
        for i in range(qc.num_qubits):
            qc.sdg(i)
            qc.h(i)
    qc.measure_all()
    return qc


def normalize_counts(counts) -> dict[str, int]:
    if isinstance(counts, list):
        if len(counts) != 1:
            raise ValueError(f"Expected one counts dict, got {len(counts)}")
        counts = counts[0]
    return {str(k): int(v) for k, v in counts.items()}


def run_circuits_on_backend(
    backend,
    circuits: list[QuantumCircuit],
    shots: int,
    optimization_level: int,
    seed_transpiler: int | None,
    max_circuits_per_job: int,
) -> list[dict[str, int]]:
    """Transpile and execute circuits; return counts in input order."""
    transpiled = [
        transpile_for_backend(qc, backend, optimization_level, seed_transpiler)
        for qc in circuits
    ]
    all_counts: list[dict[str, int]] = []
    batch_size = max(1, max_circuits_per_job)
    for start in range(0, len(transpiled), batch_size):
        batch = transpiled[start : start + batch_size]
        result = backend.run(batch, shots=shots).result()
        counts_list = result.get_counts()
        if not isinstance(counts_list, list):
            counts_list = [counts_list]
        if len(counts_list) != len(batch):
            raise RuntimeError(
                f"Expected {len(batch)} count dicts, backend returned {len(counts_list)}"
            )
        all_counts.extend(normalize_counts(c) for c in counts_list)
    return all_counts


def estimate_mw_from_hardware_counts(
    counts_by_basis: dict[BasisName, dict[str, int]],
    n_qubits: int,
) -> tuple[float, list[float], list[float], list[float]]:
    x_exp = [
        qubit_expectation_from_counts(counts_by_basis["X"], i, n_qubits)
        for i in range(n_qubits)
    ]
    y_exp = [
        qubit_expectation_from_counts(counts_by_basis["Y"], i, n_qubits)
        for i in range(n_qubits)
    ]
    z_exp = [
        qubit_expectation_from_counts(counts_by_basis["Z"], i, n_qubits)
        for i in range(n_qubits)
    ]
    score = mw_score_from_bloch(x_exp, y_exp, z_exp)
    return score, x_exp, y_exp, z_exp


def total_circuit_jobs(n_ansatzes: int, n_depths: int, n_samples: int) -> int:
    return n_ansatzes * n_depths * n_samples * len(BASIS_ORDER)


def compute_iqm_mw_scores(
    backend,
    ansatz_fn: Callable[[int, int], QuantumCircuit],
    n_qubits: int,
    depth: int,
    n_samples: int,
    seed: int,
    shots: int,
    optimization_level: int,
    seed_transpiler: int | None,
    max_circuits_per_job: int,
) -> dict[str, object]:
    qc = ansatz_fn(n_qubits, depth)
    params = list(qc.parameters)
    rng = np.random.default_rng(seed)

    scores: list[float] = []
    bloch_rows: list[dict[str, float]] = []
    pending_circuits: list[QuantumCircuit] = []
    pending_sample_indices: list[int] = []

    def flush_batch() -> None:
        nonlocal pending_circuits, pending_sample_indices
        if not pending_circuits:
            return
        counts_list = run_circuits_on_backend(
            backend,
            pending_circuits,
            shots=shots,
            optimization_level=optimization_level,
            seed_transpiler=seed_transpiler,
            max_circuits_per_job=max_circuits_per_job,
        )
        grouped = _group_counts_by_sample(counts_list)
        if len(grouped) != len(pending_sample_indices):
            raise RuntimeError(
                f"Batch mismatch: {len(grouped)} sample groups vs "
                f"{len(pending_sample_indices)} sample indices"
            )
        for sample_index, basis_counts in zip(pending_sample_indices, grouped):
            score, x_exp, y_exp, z_exp = estimate_mw_from_hardware_counts(
                basis_counts, n_qubits
            )
            scores.append(score)
            row: dict[str, float] = {"sample_index": float(sample_index), "mw_score": score}
            for i in range(n_qubits):
                row[f"x_q{i}"] = x_exp[i]
                row[f"y_q{i}"] = y_exp[i]
                row[f"z_q{i}"] = z_exp[i]
            bloch_rows.append(row)
        pending_circuits = []
        pending_sample_indices = []

    for sample_index in range(n_samples):
        values = rng.uniform(0.0, 2.0 * math.pi, size=len(params))
        bound = qc.assign_parameters(dict(zip(params, values)), inplace=False)
        for basis in BASIS_ORDER:
            pending_circuits.append(add_basis_measurement(bound, basis))
        pending_sample_indices.append(sample_index)
        if len(pending_circuits) >= max_circuits_per_job:
            flush_batch()
    flush_batch()

    if len(scores) != n_samples:
        raise RuntimeError(f"Expected {n_samples} MW scores, got {len(scores)}")

    arr = np.array(scores, dtype=float)
    return {
        "mw_scores": scores,
        "bloch_rows": bloch_rows,
        "mw_avg": float(np.mean(arr)),
        "mw_std": float(np.std(arr)),
        "mw_sem": float(np.std(arr) / math.sqrt(n_samples)),
        "mw_min": float(np.min(arr)),
        "mw_max": float(np.max(arr)),
        "depth": depth,
        "n_qubits": n_qubits,
        "n_params": len(params),
        "n_samples": n_samples,
    }


def _group_counts_by_sample(counts_list: list[dict[str, int]]) -> list[dict[BasisName, dict[str, int]]]:
    if len(counts_list) % len(BASIS_ORDER) != 0:
        raise ValueError("Counts list length must be a multiple of 3 (Z, X, Y per sample)")
    grouped: list[dict[BasisName, dict[str, int]]] = []
    for i in range(0, len(counts_list), len(BASIS_ORDER)):
        grouped.append(
            {
                "Z": counts_list[i],
                "X": counts_list[i + 1],
                "Y": counts_list[i + 2],
            }
        )
    return grouped


def verify_count_expectation_helper() -> None:
    n = 5
    counts_all_zero = {"00000": 1000}
    for q in range(n):
        assert abs(qubit_expectation_from_counts(counts_all_zero, q, n) - 1.0) < 1e-12

    counts_q0_one = {"00001": 1000}
    assert abs(qubit_expectation_from_counts(counts_q0_one, 0, n) - (-1.0)) < 1e-12
    assert abs(qubit_expectation_from_counts(counts_q0_one, 1, n) - 1.0) < 1e-12

    counts_by_basis: dict[BasisName, dict[str, int]] = {
        "Z": counts_all_zero,
        "X": counts_all_zero,
        "Y": counts_all_zero,
    }
    assert abs(estimate_mw_from_hardware_counts(counts_by_basis, n)[0]) < 1e-12


def verify_bloch_mw_matches_statevector() -> None:
    pauli_x = np.array([[0, 1], [1, 0]], dtype=complex)
    pauli_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
    pauli_z = np.array([[1, 0], [0, -1]], dtype=complex)

    n = 5
    rng = np.random.default_rng(0)
    for _ in range(20):
        psi = rng.normal(size=2**n) + 1j * rng.normal(size=2**n)
        psi /= np.linalg.norm(psi)
        sv_mw = meyer_wallach_score(psi, n)
        x_exp: list[float] = []
        y_exp: list[float] = []
        z_exp: list[float] = []
        for i in range(n):
            rho_i = single_qubit_reduced_density(psi, i, n)
            x_exp.append(float(np.real(np.trace(rho_i @ pauli_x))))
            y_exp.append(float(np.real(np.trace(rho_i @ pauli_y))))
            z_exp.append(float(np.real(np.trace(rho_i @ pauli_z))))
        bloch_mw = mw_score_from_bloch(x_exp, y_exp, z_exp)
        assert abs(sv_mw - bloch_mw) < 1e-9, f"sv={sv_mw}, bloch={bloch_mw}"


def run_self_check() -> None:
    verify_mw_implementation()
    verify_count_expectation_helper()
    verify_bloch_mw_matches_statevector()
    print("All self-checks passed.")




## 4. Self-check (no hardware)

In [ ]:
run_self_check()

## 5. Connect to IQM Spark

Set `IQM_TOKEN` in your environment or enter the token when prompted.


In [ ]:
iqm_backend = connect_to_iqm_backend(IQM_URL)
print(f"Connected to backend: {iqm_backend}  (n_qubits = {iqm_backend.num_qubits})")


## 6. Hardware sweep

In [ ]:
stamp = datetime.now(tz=timezone.utc).strftime("%Y%m%d_%H%M%S")
output_dir = NOTEBOOK_DIR / f"iqm_mw_{stamp}"
output_dir.mkdir(parents=True, exist_ok=True)

ansatz_fns = {
    "ansatz_odra": ansatz_odra,
    "ansatz_simulator": ansatz_simulator,
}

summary_rows = []
score_rows = []

for depth in DEPTHS:
    for ansatz_name in ANSATZES:
        print(f"Running {ansatz_name} depth={depth} ...")
        depth_seed = SEED + depth * 1000 + (1 if ansatz_name == "ansatz_simulator" else 0)
        result = compute_iqm_mw_scores(
            iqm_backend,
            ansatz_fns[ansatz_name],
            n_qubits=NUM_QUBITS,
            depth=depth,
            n_samples=N_SAMPLES,
            seed=depth_seed,
            shots=SHOTS,
            optimization_level=OPTIMIZATION_LEVEL,
            seed_transpiler=None,
            max_circuits_per_job=MAX_CIRCUITS_PER_JOB,
        )
        summary_rows.append(
            {
                "ansatz": ansatz_name,
                "depth": depth,
                "n_qubits": result["n_qubits"],
                "n_params": result["n_params"],
                "n_samples": result["n_samples"],
                "shots": SHOTS,
                "seed": depth_seed,
                "mw_avg": result["mw_avg"],
                "mw_std": result["mw_std"],
                "mw_sem": result["mw_sem"],
                "mw_min": result["mw_min"],
                "mw_max": result["mw_max"],
            }
        )
        for bloch in result["bloch_rows"]:
            row = {
                "ansatz": ansatz_name,
                "depth": depth,
                "sample_index": int(bloch["sample_index"]),
                "mw_score": bloch["mw_score"],
            }
            for key, value in bloch.items():
                if key not in ("sample_index", "mw_score"):
                    row[key] = value
            score_rows.append(row)

summary_path = output_dir / "iqm_mw_results.csv"
scores_path = output_dir / "iqm_mw_scores.csv"
manifest_path = output_dir / "run_manifest.json"

with summary_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary_rows[0]))
    writer.writeheader()
    writer.writerows(summary_rows)

score_fieldnames = list(score_rows[0].keys()) if score_rows else []
with scores_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=score_fieldnames)
    writer.writeheader()
    writer.writerows(score_rows)

manifest = {
    "created_utc": datetime.now(tz=timezone.utc).isoformat(),
    "backend": str(iqm_backend),
    "iqm_url": IQM_URL,
    "source_notebook": "evaluation_and_comparison/iqm_meyer_wallach.ipynb",
    "method": "local_xyz_tomography",
    "n_qubits": NUM_QUBITS,
    "depths": list(DEPTHS),
    "ansatzes": list(ANSATZES),
    "n_samples": N_SAMPLES,
    "shots": SHOTS,
    "seed": SEED,
    "optimization_level": OPTIMIZATION_LEVEL,
    "max_circuits_per_job": MAX_CIRCUITS_PER_JOB,
    "total_circuits": n_jobs,
    "outputs": [summary_path.name, scores_path.name],
}
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")

results_df = pd.DataFrame(summary_rows)
scores_df = pd.DataFrame(score_rows)
print(f"\nSaved outputs to {output_dir}")
results_df


## 7. Results & Plots

In [ ]:
print("MW comparison (higher = more entanglement):")
print("depth | ansatz           | mw_avg   | mw_std")
print("-" * 50)
for _, row in results_df.iterrows():
    print(
        f"{int(row['depth']):>5} | {row['ansatz']:<16} | "
        f"{row['mw_avg']:.6f} | {row['mw_std']:.6f}"
    )

fig, ax = plt.subplots(figsize=(8, 4))
for ansatz_name in ANSATZES:
    sub = results_df[results_df["ansatz"] == ansatz_name].sort_values("depth")
    ax.plot(sub["depth"], sub["mw_avg"], marker="o", label=ansatz_name)
    ax.fill_between(
        sub["depth"],
        sub["mw_avg"] - sub["mw_sem"],
        sub["mw_avg"] + sub["mw_sem"],
        alpha=0.2,
    )
ax.set_xlabel("Depth")
ax.set_ylabel("Mean Meyer–Wallach score")
ax.set_title("MW entanglement vs depth on IQM Spark")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
